# Lab: Build a RAG Application from Scratch

Retrieval-Augmented Generation in ~80 lines, no framework: chunk a knowledge
base, index it, retrieve for a question, assemble a grounded prompt, and
generate an answer with citations — then evaluate it.

The LLM is a deterministic stand-in (`DemoLLM`) with the same interface as a
real client, so the lab runs offline; the final exercise swaps in a real
provider.

In [1]:
# The knowledge base for every lab this week: support documents for Atlas
# Cycles, a fictional e-bike maker. Small enough to read, real enough to
# retrieve against.
CORPUS = {
    "battery-care": (
        "Atlas S2 battery care. Charge the battery to 80 percent for daily "
        "use and only to 100 percent before a long ride. Store between 10 "
        "and 25 degrees Celsius. A full recharge takes 4.5 hours from empty."
    ),
    "warranty": (
        "Atlas warranty policy. The frame is covered for 5 years. The "
        "battery and motor are covered for 2 years or 15,000 km, whichever "
        "comes first. Wear parts such as brake pads and tires are excluded."
    ),
    "range": (
        "Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km "
        "in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo "
        "weight, and cold weather reduce range by up to 30 percent."
    ),
    "error-codes": (
        "Atlas display error codes. E01 means a motor sensor fault: restart "
        "the system. E04 means battery communication lost: reseat the "
        "battery. E09 means brake cutoff engaged: check the brake levers."
    ),
    "first-service": (
        "First service. Book the complimentary first service after 300 km "
        "or 3 months. Spoke tension, brake bedding, and firmware updates "
        "are included at no charge."
    ),
}

print(f"knowledge base loaded: {len(CORPUS)} documents")

knowledge base loaded: 5 documents


## Step 1 — Chunking

Retrieval works on passages. Whole documents dilute the vector (many topics,
one vector); sentences are too small to answer from. We chunk at sentence
boundaries with a size target — crude, but the trade-off is the real one.

In [2]:
def chunk(doc_id, text, target_words=28):
    # Split on sentence boundaries (". "), not every period — "4.5 hours"
    # must survive chunking intact.
    sentences = [s.strip() + "." for s in text.rstrip(".").split(". ") if s.strip()]
    chunks, current = [], []
    for s in sentences:
        current.append(s)
        if sum(len(x.split()) for x in current) >= target_words:
            chunks.append((f"{doc_id}#{len(chunks)}", " ".join(current)))
            current = []
    if current:
        chunks.append((f"{doc_id}#{len(chunks)}", " ".join(current)))
    return chunks

CHUNKS = [c for doc_id, text in CORPUS.items() for c in chunk(doc_id, text)]
for cid, text in CHUNKS[:4]:
    print(f"{cid:<16} {text[:60]}...")
print(f"... {len(CHUNKS)} chunks total")

battery-care#0   Atlas S2 battery care. Charge the battery to 80 percent for ...
battery-care#1   A full recharge takes 4.5 hours from empty....
warranty#0       Atlas warranty policy. The frame is covered for 5 years. The...
range#0          Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to...
... 6 chunks total


## Step 2 — Index and retrieve

Same embedding + cosine machinery as the lecture, now over chunks.

In [3]:
# A deterministic embedding: character trigram counts. No model, no network,
# yet it captures enough word-shape overlap to demonstrate the geometry that
# real embedding models learn.
from collections import Counter
import math

def embed(text):
    t = " " + "".join(c.lower() if c.isalnum() else " " for c in text) + " "
    return Counter(t[i:i+3] for i in range(len(t) - 2) if t[i:i+3].strip())

def cosine(a, b):
    dot = sum(a[k] * b[k] for k in a.keys() & b.keys())
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

INDEX = [(cid, embed(text), text) for cid, text in CHUNKS]

def retrieve(query, k=3):
    q = embed(query)
    ranked = sorted(((cosine(q, v), cid, text) for cid, v, text in INDEX),
                    reverse=True)
    return ranked[:k]

for score, cid, text in retrieve("how long is the battery covered"):
    print(f"{score:.3f}  {cid:<16} {text[:55]}...")

0.354  warranty#0       Atlas warranty policy. The frame is covered for 5 years...
0.321  error-codes#0    Atlas display error codes. E01 means a motor sensor fau...
0.292  battery-care#0   Atlas S2 battery care. Charge the battery to 80 percent...


## Step 3 — The model seam

`DemoLLM` is deliberately boring: given a context block and a question, it
answers extractively from the highest-scoring sentence and cites the chunk
it used. A real client (OpenAI, Anthropic, a local llama.cpp server) drops
into the same `complete(prompt)` seam — that interface is the design point.

In [4]:
class DemoLLM:
    # Deterministic stand-in. Same seam as a real client: complete(prompt).
    def complete(self, prompt):
        context, _, question = prompt.partition("QUESTION:")
        question = question.strip().lower()
        best, best_score = None, -1.0
        for line in context.splitlines():
            if "|" not in line:
                continue
            cid, sentence = line.split("|", 1)
            score = cosine(embed(question), embed(sentence))
            if score > best_score:
                best, best_score = (cid.strip(), sentence.strip()), score
        if best is None or best_score < 0.05:
            return "I don't know based on the provided context."
        cid, sentence = best
        return f"{sentence} [source: {cid}]"

llm = DemoLLM()
print(llm.complete("k1 | The sky is blue on clear days.\nQUESTION: what color is the sky"))

The sky is blue on clear days. [source: k1]


## Step 4 — Assemble the pipeline

RAG is just these pieces in order: retrieve, build a grounded prompt,
generate. Note what the prompt *forbids* — answering beyond the context.
That single instruction is the difference between grounded and hallucinated.

In [5]:
def rag_answer(question, k=3):
    hits = retrieve(question, k)
    context = "\n".join(f"{cid} | {text}" for _, cid, text in hits)
    prompt = (
        "Answer ONLY from the context below. If the context does not "
        "contain the answer, say you don't know.\n\n"
        f"{context}\n\nQUESTION: {question}"
    )
    return llm.complete(prompt), [cid for _, cid, _ in hits]

for q in [
    "how long is the battery under warranty",
    "what does error code E04 mean",
    "can I ride 100 km in Boost mode",
]:
    answer, sources = rag_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   retrieved: {sources}")
    print()

Q: how long is the battery under warranty
A: Atlas S2 battery care. Charge the battery to 80 percent for daily use and only to 100 percent before a long ride. Store between 10 and 25 degrees Celsius. [source: battery-care#0]
   retrieved: ['battery-care#0', 'error-codes#0', 'warranty#0']

Q: what does error code E04 mean
A: Atlas display error codes. E01 means a motor sensor fault: restart the system. E04 means battery communication lost: reseat the battery. E09 means brake cutoff engaged: check the brake levers. [source: error-codes#0]
   retrieved: ['error-codes#0', 'warranty#0', 'range#0']

Q: can I ride 100 km in Boost mode
A: Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo weight, and cold weather reduce range by up to 30 percent. [source: range#0]
   retrieved: ['range#0', 'first-service#0', 'battery-care#0']



## Step 5 — Evaluate before you ship

A vibe check is a start; a *repeatable* vibe check is an eval. Ten seconds
of assertion loop catches the regressions that re-chunking or a new
embedding model will someday introduce.

In [6]:
EVAL_SET = [
    ("how long are the battery and motor covered", "2 years"),
    ("what does error code E04 mean", "reseat"),
    ("when is the first service due", "300 km"),
    ("how long does a full recharge take", "4.5 hours"),
    ("does the warranty cover brake pads", "excluded"),
]

passed = 0
for question, must_contain in EVAL_SET:
    answer, _ = rag_answer(question)
    ok = must_contain.lower() in answer.lower()
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {question!r} -> expects {must_contain!r}")
print(f"\n{passed}/{len(EVAL_SET)} passed")

PASS  'how long are the battery and motor covered' -> expects '2 years'
PASS  'what does error code E04 mean' -> expects 'reseat'
PASS  'when is the first service due' -> expects '300 km'
PASS  'how long does a full recharge take' -> expects '4.5 hours'
PASS  'does the warranty cover brake pads' -> expects 'excluded'

5/5 passed


## Exercise — make it real

1. Replace `DemoLLM.complete` with a real client call (the program-issued
   key loads from `.env`; never paste it into a cell).
2. Re-run the eval set. Watch for the failure mode the stand-in cannot
   exhibit: a *fluent* answer when retrieval missed — hallucination. Add an
   eval case whose answer is **not** in the knowledge base and assert the
   model says it doesn't know.

That failure mode — and how to instrument for it — is where the next
session picks up.